# 04 Set style vs pressure
Step 7: does how a lineman sets relate to pressure allowed, beyond size and situation?

Design (Claude's calls, approved 10/27 "go with ur gut"):
- **Outcome:** `pressure_allowed` (main), `peak_pressure_probability_allowed` (check).
- **Set style is player-level.** Each player's average set features, so the set on a play can't simply be the pressure on that play showing up early.
- **Split-sample.** Each player's plays are split at random. Set style comes from half A, and pressure is modeled on half B, so the same plays never sit on both sides.
- **Leakage:** plays with pressure inside the 1.5 s window are dropped.
- **Baseline:** Combine weight, position, dropback duration, rushers encountered. Draft slot excluded.
- **Model:** logistic regression, standard errors clustered by player, plus 5-fold cross-validation with whole players held out.
- Repeated over 20 random splits so one split doesn't drive the result.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import log_loss, roc_auc_score
from shared.config import PROCESSED

warnings.filterwarnings('ignore')
CLEAN = PROCESSED / 'clean'
RAW_PQ = PROCESSED / 'parquet'
KEYS = ['game_id', 'play_id', 'nfl_id']
FEATS = ['retreat_depth', 'lateral_shift', 'peak_speed', 'peak_accel_05']

## Game set features per play
Same as 03: 1.5 s window, regular season, pass blocking, OL.

In [ ]:
win = pd.read_parquet(CLEAN / 'ol_passset_window.parquet')
plays = pd.read_parquet(CLEAN / 'player_play.parquet',
                        columns=KEYS + ['dropback_duration', 'pass_rushers_encountered', 'pressure_allowed',
                                        'time_to_pressure_allowed', 'peak_pressure_probability_allowed'])
players = pd.read_parquet(RAW_PQ / 'nfl-big-data-bowl-2027__players.parquet', columns=['nfl_id', 'nfl_position'])
cr = pd.read_parquet(RAW_PQ / 'nfl-big-data-bowl-2027__combine_results.parquet', columns=['nfl_id', 'combine_weight'])

stance = win[win['t'] < 0].groupby(KEYS)[['x', 'y']].mean()
end = win[win['t'] == 1.5].set_index(KEYS)[['x', 'y']]
game = pd.DataFrame({
    'retreat_depth': stance['x'] - end['x'],
    'lateral_shift': (end['y'] - stance['y']).abs(),
    'peak_speed': win[win['t'].between(0, 1.5)].groupby(KEYS)['s_sg'].max(),
    'peak_accel_05': win[win['t'].between(0, 0.5)].groupby(KEYS)['a_sg'].max(),
}).reset_index().merge(plays, on=KEYS).merge(players, on='nfl_id').merge(cr, on='nfl_id')
game['y'] = game['pressure_allowed'].astype(float)
len(game), game['y'].mean().round(3)

Leakage check: how many pressures land inside the window?

In [ ]:
inside = game['time_to_pressure_allowed'] <= 1.5
inside.sum(), inside[game['y'] == 1].mean().round(3)

In [ ]:
game = game[~inside]
n_plays = game.groupby('nfl_id').size()
game = game[game['nfl_id'].isin(n_plays[n_plays >= 100].index)].copy()
game['rush2'] = (game['pass_rushers_encountered'] >= 2).astype(int)
game['rush0'] = (game['pass_rushers_encountered'] == 0).astype(int)
game['w_z'] = (game['combine_weight'] - game['combine_weight'].mean()) / game['combine_weight'].std()
len(game), game['nfl_id'].nunique()

## One split, shown in full
Set style = player's mean of each feature in half A, as a z-score across players (1 = one SD more than the typical lineman).

In [ ]:
BASE = 'y ~ w_z + C(nfl_position) + dropback_duration + rush2 + rush0'
FULL = BASE + ' + ' + ' + '.join(FEATS)

def split(game, seed):
    rng = np.random.default_rng(seed)
    half_a = rng.random(len(game)) < 0.5
    style = game[half_a].groupby('nfl_id')[FEATS].mean()
    style = (style - style.mean()) / style.std()
    b = game[~half_a].drop(columns=FEATS).merge(style, left_on='nfl_id', right_index=True)
    return b.reset_index(drop=True), style

b, style = split(game, seed=0)
fit = smf.logit(FULL, b).fit(disp=0, cov_type='cluster', cov_kwds={'groups': b['nfl_id']})
fit.summary2().tables[1][['Coef.', 'Std.Err.', 'P>|z|', '[0.025', '0.975]']].round(3)

Cross-validation with whole players held out. If set style helps, the full model's log loss is lower than the baseline's.

In [ ]:
def cv_compare(b, seed, k=5):
    rng = np.random.default_rng(seed)
    ids = b['nfl_id'].unique()
    rng.shuffle(ids)
    pred = {'base': np.zeros(len(b)), 'full': np.zeros(len(b))}
    for fold in np.array_split(ids, k):
        test = b['nfl_id'].isin(fold).to_numpy()
        pred['base'][test] = smf.logit(BASE, b[~test]).fit(disp=0).predict(b[test])
        pred['full'][test] = smf.logit(FULL, b[~test]).fit(disp=0).predict(b[test])
    return pd.DataFrame({m: {'log loss': log_loss(b['y'], p), 'AUC': roc_auc_score(b['y'], p)}
                         for m, p in pred.items()})

cv_compare(b, seed=0).round(4)

## Repeat over 20 splits

In [ ]:
coefs, gaps = [], []
for seed in range(20):
    b, _ = split(game, seed)
    coefs.append(smf.logit(FULL, b).fit(disp=0).params[FEATS])
    cv = cv_compare(b, seed)
    gaps.append(cv.loc['log loss', 'full'] - cv.loc['log loss', 'base'])

print('CV log loss, full minus base (positive = set style made it worse):',
      round(np.mean(gaps), 5), 'range', round(min(gaps), 5), 'to', round(max(gaps), 5))
pd.DataFrame(coefs).agg(['mean', 'std']).round(3)

## Player level, within position, no situation controls
Raw pattern for context: each player's pressure rate in half B against his set style from half A, both centered within position.

In [ ]:
b, style = split(game, seed=0)
pl = b.groupby('nfl_id').agg(rate=('y', 'mean'), pos=('nfl_position', 'first')).join(style)
for c in ['rate'] + FEATS:
    pl[c] = pl[c] - pl.groupby('pos')[c].transform('mean')
pd.Series({c: np.corrcoef(pl['rate'], pl[c])[0, 1] for c in FEATS}).round(2)

## Check: continuous outcome
Peak pressure probability allowed, same model as OLS, one split.

In [ ]:
b2 = b.dropna(subset=['peak_pressure_probability_allowed'])
ols = smf.ols(FULL.replace('y ~', 'peak_pressure_probability_allowed ~'), b2).fit(
    cov_type='cluster', cov_kwds={'groups': b2['nfl_id'].to_numpy()})
ols.summary2().tables[1].loc[FEATS, ['Coef.', 'Std.Err.', 'P>|z|']].round(4)

## Notes
Interpretation goes in `decisions.md`.